Đọc dữ liệu

In [1]:
# Nạp thư viện os (Operating System) — dùng để tương tác với hệ điều hành (ví dụ: quản lý tệp tin, nối đường dẫn thư mục, kiểm tra tệp tồn tại).
import os
# Nạp thư viện numpy và đặt tên viết tắt là np — dùng để xử lý tính toán đại số tuyến tính, mảng nhiều chiều (array) và các phép toán số học tốc độ cao.
import numpy as np
# Nạp thư viện pandas và đặt tên viết tắt là pd — dùng để làm việc với dữ liệu dạng bảng (DataFrame), hỗ trợ đọc/ghi tệp CSV, Excel và tiền xử lý dữ liệu
import pandas as pd

# Ghép đường dẫn giữa thư mục 'path' (từ kagglehub) và tên tệp CSV
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc tệp CSV vào biến 'diamonds' (DataFrame) với dấu phân cách các cột là dấu phẩy (,)
diamonds = pd.read_csv(file_path, sep=',', index_col=0)

# In kích thước của tập dữ liệu (số dòng, số cột)
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

# In tiêu đề thông báo cho phần hiển thị 5 dòng đầu
print("\n--- 5 dòng đầu tiên ---")
# Hiển thị 5 dòng dữ liệu đầu tiên
print(diamonds.head())

# In tiêu đề thông báo cho phần kiểm tra các giá trị phân loại
print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
# Lấy và in danh sách các giá trị duy nhất (không trùng lặp) của cột 'cut' (vết cắt)
print(f"Cut: {diamonds['cut'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'color' (màu sắc)
print(f"Color: {diamonds['color'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'clarity' (độ tinh khiết)
print(f"Clarity: {diamonds['clarity'].unique()}")

# In tiêu đề thông báo cho phần thống kê mô tả
print("\n--- Thống kê mô tả ---")
# Tính toán các chỉ số thống kê cơ bản cho các cột số
print(diamonds.describe())

Kích thước tập dữ liệu ban đầu: (53943, 10)

<class 'pandas.DataFrame'>
RangeIndex: 53943 entries, 1 to 53943
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53943 non-null  float64
 1   cut      53943 non-null  str    
 2   color    53943 non-null  str    
 3   clarity  53943 non-null  str    
 4   depth    53943 non-null  float64
 5   table    53943 non-null  float64
 6   price    53943 non-null  int64  
 7   x        53943 non-null  float64
 8   y        53943 non-null  float64
 9   z        53943 non-null  float64
dtypes: float64(6), int64(1), str(3)
memory usage: 4.1 MB

--- 5 dòng đầu tiên ---
   carat      cut color clarity  depth  table  price     x     y     z
1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
3   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
4   0.29  Premium     I     VS2   

Tiền xử lý

In [2]:
# Nạp công cụ OrdinalEncoder từ thư viện scikit-learn để mã hóa các biến phân loại có thứ tự
from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
# Kiểm tra xem cột số thứ tự thừa 'Unnamed: 0' có tồn tại trong DataFrame hay không
if 'Unnamed: 0' in diamonds.columns:
    # Nếu có, xóa bỏ cột này khỏi tập dữ liệu
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (kích thước x, y, z <= 0)
# Lấy tổng số dòng dữ liệu ban đầu trước khi lọc
initial_rows = len(diamonds)

# Lọc giữ lại các dòng có cả 3 kích thước (x, y, z) đều lớn hơn 0 và tạo bản sao dữ liệu mới
diamonds = diamonds[(diamonds['x'] > 0) & (diamonds['y'] > 0) & (diamonds['z'] > 0)].copy()

# Tính số lượng dòng dữ liệu bị loại bỏ (bằng số dòng ban đầu trừ số dòng sau khi lọc)
invalid_dimensions = initial_rows - len(diamonds)

# In số lượng dòng nhiễu đã bị loại bỏ ra màn hình (thêm \n ở đầu để cách dòng trên)
print(f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu (x, y, z <= 0).")

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
# Đếm tổng số dòng dữ liệu bị lặp lại hoàn toàn trong tập dữ liệu
num_duplicates = diamonds.duplicated().sum()

# Xóa bỏ các dòng trùng lặp, chỉ giữ lại 1 bản ghi đại diện
diamonds = diamonds.drop_duplicates()

# In số lượng dòng trùng lặp đã xóa ra màn hình (thêm \n ở đầu để cách với dòng trên)
print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding) cho các thuộc tính phân loại
# Định nghĩa danh sách các mức độ chất lượng của vết cắt 'cut' theo thứ tự tăng dần (Fair là thấp nhất, Ideal là cao nhất)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

# Định nghĩa danh sách màu sắc 'color' theo thứ tự tăng dần từ kém nhất đến tốt nhất (J -> D)
color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

# Định nghĩa danh sách độ tinh khiết 'clarity' theo thứ tự tăng dần từ kém nhất đến tốt nhất (I1 -> IF)
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Khởi tạo đối tượng OrdinalEncoder với danh sách thứ tự đã thiết lập cho từng cột
encoder = OrdinalEncoder(categories=[cut_categories, color_categories, clarity_categories])

# chuyển đổi các cột dữ liệu dạng chữ (phân loại) thành dạng số dựa trên thứ tự đã định nghĩa
diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(diamonds[['cut', 'color', 'clarity']])

# 5: Kiểm tra dữ liệu sau tiền xử lý
# In kích thước mới của tập dữ liệu (số dòng, số cột) sau khi đã làm sạch và chuyển đổi
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

# In tiêu đề thông báo hiển thị 5 dòng đầu
print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")

# Hiển thị 5 dòng dữ liệu đầu tiên sau tiền xử lý
print(diamonds.head())


Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
1   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
2   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
3   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
4   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
5   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


Chia tập dữ liệu

In [3]:
from sklearn.model_selection import train_test_split

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])
# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu train test
X_train, X_test, y_train, y_test = train_test_split(
      X,
      y,
      test_size=0.2,
      random_state=42
    )

# In ra kết quả chia dữ liệu
print(f"Có {X_train.shape[0]} mẫu huấn luyện và {X_test.shape[0]} mẫu kiểm tra")

Có 43020 mẫu huấn luyện và 10755 mẫu kiểm tra


Training

In [4]:

# Import các thư viện cần thiết
# Import mô hình Support Vector Regression
from sklearn.svm import SVR  
# Import công cụ chuẩn hóa dữ liệu
from sklearn.preprocessing import StandardScaler  
# Kết hợp chuẩn hóa và mô hình thành một quy trình
from sklearn.pipeline import make_pipeline          
# Thư viện tính toán số học
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Các chỉ số đánh giá
import numpy as np                                  
# Thư viện xử lý dữ liệu
import pandas as pd                                 
# Thư viện đo thời gian
import time                                         


# X là dữ liệu các thuộc tính đầu vào
X = diamonds.drop(columns=["price"])

# y là dữ liệu nhãn cần dự đoán
y = diamonds["price"]

# Chia dữ liệu thành tập huấn luyện (80%) và tập kiểm tra (20%)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% dữ liệu dùng để kiểm tra
    random_state=42      # Cố định kết quả chia dữ liệu
)

# Khởi tạo mô hình SVR kết hợp chuẩn hóa dữ liệu
svr_model = make_pipeline(
    StandardScaler(),    # Chuẩn hóa các thuộc tính về cùng thang đo
    SVR(
        kernel="rbf",    # Sử dụng kernel RBF để xử lý mối quan hệ phi tuyến
        C=100,           # Tham số điều chỉnh mức phạt lỗi dự đoán
        epsilon=0.1,     # Khoảng sai số cho phép trong SVR
        gamma="scale"    # Tự động xác định hệ số ảnh hưởng của các điểm dữ liệu
    )
)

# Bắt đầu tính thời gian huấn luyện
start_time = time.time()

# Huấn luyện mô hình SVR bằng tập dữ liệu huấn luyện
svr_model.fit(X_train, y_train)

# Kết thúc tính thời gian huấn luyện
training_time = time.time() - start_time

#Dự đoán giá kim cương trên tập huấn luyện để đánh giá hiệu suất mô hình
y_pred_train = svr_model.predict(X_train)
# Sử dụng mô hình đã huấn luyện để dự đoán giá kim cương trên tập kiểm tra
y_pred_test = svr_model.predict(X_test)

# Tính toán các chỉ số đánh giá trên tập huấn luyện
mae_train = mean_absolute_error(y_train, y_pred_train)  # Sai số tuyệt đối trung bình
mse_train = mean_squared_error(y_train, y_pred_train)   # Sai số bình phương trung bình
rmse_train = np.sqrt(mse_train)                          # Căn bậc hai của  MSE
r2_train = r2_score(y_train, y_pred_train)               # Hệ số    xác định R²         

# Tính toán các chỉ số đánh giá trên tập kiểm tra
mae_test = mean_absolute_error(y_test, y_pred_test)     # Sai số tuyệt đối trung bình
mse_test = mean_squared_error(y_test, y_pred_test)      # Sai số bình   phương trung bình       
rmse_test = np.sqrt(mse_test)                            # Căn bậc hai của  MSE
r2_test = r2_score(y_test, y_pred_test)                  # Hệ số xác định R²


# In các chỉ số đánh giá trên Train
print(f"MAE Train: {mae_train:.4f}")
print(f"MSE Train: {mse_train:.4f}")
print(f"RMSE Train: {rmse_train:.4f}")
print(f"R² Train: {r2_train:.4f}")

# In các chỉ số đánh giá trên Test
print(f"MAE Test: {mae_test:.4f}")
print(f"MSE Test: {mse_test:.4f}")
print(f"RMSE Test: {rmse_test:.4f}")
print(f"R² Test: {r2_test:.4f}")

# In thời gian huấn luyện
print(f"Thời gian Train: {training_time:.4f} giây")

MAE Train: 350.3749
MSE Train: 529048.1847
RMSE Train: 727.3570
R² Train: 0.9667
MAE Test: 359.2113
MSE Test: 559653.7796
RMSE Test: 748.1001
R² Test: 0.9646
Thời gian Train: 70.2835 giây
